# Reference checker

Extracts the reference list of a PDF with GROBID and looks each reference up in OpenAlex, Crossref, PubMed and Europe PMC. Every reference gets a similarity score (title, first author, year) against the best candidate from each source, a verdict, and the matched record's identifiers. The result is written to `reference_check_results.csv`.

Written for Google Colab: run the cells in order. The first cell installs Java 21, builds the GROBID service from source and starts it in the background, which takes several minutes. Set your e-mail address in the `mailto` and `email` arguments; the APIs use it to identify polite clients.

In [ ]:
# Java 21
!apt-get -q update
!apt-get -q install -y openjdk-21-jdk-headless

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

!java -version

# GROBID (service only)

# remove a previous checkout
!rm -rf grobid

# clone the source
!git clone https://github.com/kermitt2/grobid.git

%cd grobid

# build and start only grobid-service; the trainer's publish/generatePom tasks are not needed,
# gradle builds the core and the dependencies the service requires
!nohup ./gradlew :grobid-service:run -Dorg.gradle.java.home=/usr/lib/jvm/java-21-openjdk-amd64 &

%cd ..

# Python packages
!pip install -q lxml rapidfuzz requests pandas


In [ ]:
import requests, time

time.sleep(15)  # give the service some time to come up

try:
    r = requests.get("http://localhost:8070/api/isalive")
    print("Status code:", r.status_code)
    print("Body:", r.text[:200])
except Exception as e:
    print("Connection error:", e)

In [ ]:
from google.colab import files

print("Upload the PDF whose references you want to check.")
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)


In [ ]:
import time
import requests

GROBID_URL = "http://localhost:8070/api/processFulltextDocument"

# a short wait in case the service is still starting
time.sleep(10)

with open(pdf_path, "rb") as f:
    files_req = {"input": f}
    params = {
        "consolidateCitations": 1,  # enrich the citations with external data where possible
        "includeRawCitations": 1,
        "includeTeiIds": 1,
    }
    print("Sending the request to GROBID...")
    r = requests.post(GROBID_URL, files=files_req, data=params)

if r.status_code != 200:
    raise RuntimeError(f"GROBID returned an error: {r.status_code}\n{r.text[:1000]}")

tei_xml = r.text
print("TEI XML length:", len(tei_xml))


In [ ]:
from lxml import etree

ns = {"tei": "http://www.tei-c.org/ns/1.0"}

root = etree.fromstring(tei_xml.encode("utf-8"))

# one biblStruct per reference inside listBibl
bibl_structs = root.xpath(".//tei:listBibl/tei:biblStruct", namespaces=ns)
print("References found:", len(bibl_structs))

references = []

for b in bibl_structs:
    # title
    title_nodes = b.xpath('.//tei:title[@level="a"]/text() | .//tei:title[not(@level)]/text()', namespaces=ns)
    title = " ".join(t.strip() for t in title_nodes if t.strip()) or None

    # authors
    authors = []
    for pers in b.xpath(".//tei:author/tei:persName", namespaces=ns):
        surname = "".join(pers.xpath("./tei:surname/text()", namespaces=ns)).strip()
        forenames = pers.xpath("./tei:forename/text()", namespaces=ns)
        given_names = " ".join(f.strip() for f in forenames if f.strip())
        full_name = " ".join([given_names, surname]).strip()
        if full_name:
            authors.append(full_name)

    # year
    year = None
    date_when = b.xpath(".//tei:monogr/tei:imprint/tei:date/@when", namespaces=ns)
    if date_when:
        year = date_when[0][:4]
    else:
        date_txt = b.xpath(".//tei:monogr/tei:imprint/tei:date/text()", namespaces=ns)
        if date_txt:
            year = date_txt[0][:4]

    # doi
    doi_nodes = b.xpath('.//tei:idno[@type="DOI"]/text()', namespaces=ns)
    doi = doi_nodes[0].strip() if doi_nodes else None

    references.append(
        {
            "title": title,
            "authors": authors,
            "year": year,
            "doi": doi,
        }
    )

print("Sample references:")
for i, ref in enumerate(references[:5], start=1):
    print(f"--- {i} ---")
    print("Title:", ref["title"])
    print("Authors:", ref["authors"])
    print("Year:", ref["year"])
    print("DOI:", ref["doi"])


In [ ]:
import json
from rapidfuzz import fuzz

OPENALEX_BASE = "https://api.openalex.org/works"

def query_openalex(ref, mailto="youremail@example.com", max_results=5):
    """Search OpenAlex for one reference. The title is required; the year, when known,
    narrows the search with filter=publication_year:YYYY."""
    if not ref.get("title"):
        return []

    params = {
        "search": ref["title"],
        "per-page": max_results,
        "mailto": mailto,
    }

    # publication_year filter when the year is known
    if ref.get("year") and str(ref["year"]).isdigit():
        y = int(ref["year"])
        params["filter"] = f"publication_year:{y}"

    try:
        resp = requests.get(OPENALEX_BASE, params=params, timeout=20)
        if resp.status_code != 200:
            print("OpenAlex error:", resp.status_code, resp.text[:200])
            return []
        data = resp.json()
        return data.get("results", [])
    except Exception as e:
        print("OpenAlex request failed:", e)
        return []


def best_match_openalex(ref, candidates):
    """Best match among the OpenAlex candidates and its score: title similarity
    (token_sort_ratio), first-author surname match and a bonus for a matching year."""
    if not candidates:
        return None, 0.0

    best = None
    best_score = 0.0

    ref_title = (ref.get("title") or "").lower()
    ref_first_author_surname = None
    if ref.get("authors"):
        ref_first_author_surname = ref["authors"][0].split()[-1].lower()

    ref_year = None
    if ref.get("year") and str(ref["year"]).isdigit():
        ref_year = int(ref["year"])

    for c in candidates:
        cand_title = (c.get("title") or "").lower()
        if not cand_title:
            continue

        # title similarity
        title_score = fuzz.token_sort_ratio(ref_title, cand_title)

        # first-author surname
        cand_auths = c.get("authorships") or []
        cand_first_surname = None
        if cand_auths:
            cand_name = (cand_auths[0]
                         .get("author", {})
                         .get("display_name", "")).strip()
            if cand_name:
                cand_first_surname = cand_name.split()[-1].lower()

        author_score = 0
        if ref_first_author_surname and cand_first_surname:
            author_score = 100 if ref_first_author_surname == cand_first_surname else 0

        # year bonus
        cand_year = c.get("publication_year")
        year_bonus = 0
        if ref_year and cand_year:
            try:
                cy = int(cand_year)
                if cy == ref_year:
                    year_bonus = 10
                elif abs(cy - ref_year) <= 1:
                    year_bonus = 5
            except:
                pass

        # weighted score
        score = 0.6 * title_score + 0.3 * author_score + year_bonus

        if score > best_score:
            best_score = score
            best = c

    return best, best_score


def categorize_score(score: float) -> str:
    """Map a score to a verdict."""
    if score >= 80:
        return "high confidence, most likely correct"
    elif score >= 60:
        return "medium confidence, minor mismatches possible"
    elif score > 0:
        return "low confidence, doubtful"
    else:
        return "no match, very doubtful"


In [ ]:
import requests
from rapidfuzz import fuzz

CROSSREF_BASE = "https://api.crossref.org/works"

def query_crossref(ref, mailto="youremail@example.com", max_results=5):
    """Search Crossref for one reference. The title is required; the year, when known,
    narrows the search with a from/until pub-date filter."""
    if not ref.get("title"):
        return []

    params = {
        "query.title": ref["title"],
        "rows": max_results,
        "mailto": mailto,
    }

    # date-range filter when the year is known
    if ref.get("year") and str(ref["year"]).isdigit():
        y = int(ref["year"])
        params["filter"] = f"from-pub-date:{y}-01-01,until-pub-date:{y}-12-31"

    try:
        resp = requests.get(CROSSREF_BASE, params=params, timeout=20)
        if resp.status_code != 200:
            print("Crossref error:", resp.status_code, resp.text[:200])
            return []
        data = resp.json()
        return (data.get("message") or {}).get("items", [])
    except Exception as e:
        print("Crossref request failed:", e)
        return []


def best_match_crossref(ref, items):
    """Best match among the Crossref items and its score: title similarity
    (token_sort_ratio), first-author surname match and a bonus for a matching year."""
    if not items:
        return None, 0.0

    best = None
    best_score = 0.0

    ref_title = (ref.get("title") or "").lower()
    ref_first_author_surname = None
    if ref.get("authors"):
        ref_first_author_surname = ref["authors"][0].split()[-1].lower()

    ref_year = None
    if ref.get("year") and str(ref["year"]).isdigit():
        ref_year = int(ref["year"])

    for c in items:
        # the title usually comes as a list
        titles = c.get("title") or []
        cand_title = (titles[0] if titles else "").lower()
        if not cand_title:
            continue

        # title similarity
        title_score = fuzz.token_sort_ratio(ref_title, cand_title)

        # first-author surname
        cand_auths = c.get("author") or []
        cand_first_surname = None
        if cand_auths:
            fam = cand_auths[0].get("family", "")
            cand_first_surname = fam.strip().lower()

        author_score = 0
        if ref_first_author_surname and cand_first_surname:
            author_score = 100 if ref_first_author_surname == cand_first_surname else 0

        # year bonus
        cand_year = None
        # issued.date-parts -> [[year, month, day], ...]
        issued = c.get("issued") or {}
        date_parts = issued.get("date-parts") or []
        if date_parts and date_parts[0]:
            cand_year = date_parts[0][0]

        year_bonus = 0
        if ref_year and cand_year:
            try:
                cy = int(cand_year)
                if cy == ref_year:
                    year_bonus = 10
                elif abs(cy - ref_year) <= 1:
                    year_bonus = 5
            except:
                pass

        # weighted score, same formula as for OpenAlex
        score = 0.6 * title_score + 0.3 * author_score + year_bonus

        if score > best_score:
            best_score = score
            best = c

    return best, best_score


In [ ]:
PUBMED_ESEARCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
PUBMED_ESUMMARY = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"

def query_pubmed(ref, email="youremail@example.com", max_results=5):
    """Search PubMed by title, first author (if any) and year."""
    if not ref.get("title"):
        return []

    # query: title[Title] AND author[Author] AND year[DP]
    terms = [f'"{ref["title"]}"[Title]']
    if ref.get("authors"):
        first_author = ref["authors"][0].split()[-1]
        terms.append(f'{first_author}[Author]')
    if ref.get("year") and str(ref["year"]).isdigit():
        terms.append(f'{ref["year"]}[DP]')

    term = " AND ".join(terms)

    params = {
        "db": "pubmed",
        "term": term,
        "retmax": max_results,
        "retmode": "json",
        "email": email,
        "tool": "reference-checker-prototype",
    }

    try:
        r = requests.get(PUBMED_ESEARCH, params=params, timeout=20)
        if r.status_code != 200:
            print("PubMed esearch error:", r.status_code, r.text[:200])
            return []
        data = r.json()
        idlist = (data.get("esearchresult") or {}).get("idlist") or []
        if not idlist:
            return []

        # basic record fields through esummary
        params_sum = {
            "db": "pubmed",
            "id": ",".join(idlist),
            "retmode": "json",
            "email": email,
            "tool": "reference-checker-prototype",
        }
        rs = requests.get(PUBMED_ESUMMARY, params_sum, timeout=20)
        if rs.status_code != 200:
            print("PubMed esummary error:", rs.status_code, rs.text[:200])
            return []

        sdata = rs.json()
        result = sdata.get("result") or {}
        # result["uids"] lists the ids; one entry per id
        items = []
        for pmid in result.get("uids", []):
            items.append(result.get(pmid))
        return items
    except Exception as e:
        print("PubMed request failed:", e)
        return []


def best_match_pubmed(ref, items):
    if not items:
        return None, 0.0

    best = None
    best_score = 0.0

    ref_title = (ref.get("title") or "").lower()
    ref_first_author_surname = None
    if ref.get("authors"):
        ref_first_author_surname = ref["authors"][0].split()[-1].lower()

    ref_year = None
    if ref.get("year") and str(ref["year"]).isdigit():
        ref_year = int(ref["year"])

    for c in items:
        cand_title = (c.get("title") or "").lower()
        if not cand_title:
            continue

        title_score = fuzz.token_sort_ratio(ref_title, cand_title)

        # authors come as ['Cladouhos TT', ...]
        cand_auths = c.get("authors") or []
        cand_first_surname = None
        if cand_auths:
            name = cand_auths[0].get("name", "")  # "Cladouhos TT"
            if name:
                cand_first_surname = name.split()[0].lower()

        author_score = 0
        if ref_first_author_surname and cand_first_surname:
            author_score = 100 if ref_first_author_surname == cand_first_surname else 0

        cand_year = None
        if "pubdate" in c:
            # e.g. "2018 Jan": the first four characters are the year
            cand_year = c["pubdate"][:4]

        year_bonus = 0
        if ref_year and cand_year and str(cand_year).isdigit():
            cy = int(cand_year)
            if cy == ref_year:
                year_bonus = 10
            elif abs(cy - ref_year) <= 1:
                year_bonus = 5

        score = 0.6 * title_score + 0.3 * author_score + year_bonus

        if score > best_score:
            best_score = score
            best = c

    return best, best_score


In [ ]:
EUROPEPMC_BASE = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

def query_europepmc(ref, max_results=5):
    """Search Europe PMC by title, first author (if any) and year."""
    if not ref.get("title"):
        return []

    terms = [f'"{ref["title"]}"']
    if ref.get("authors"):
        first_author = ref["authors"][0].split()[-1]
        terms.append(first_author)
    if ref.get("year") and str(ref["year"]).isdigit():
        terms.append(ref["year"])

    query = " AND ".join(terms)

    params = {
        "query": query,
        "format": "json",
        "pageSize": max_results,
    }

    try:
        r = requests.get(EUROPEPMC_BASE, params=params, timeout=20)
        if r.status_code != 200:
            print("Europe PMC error:", r.status_code, r.text[:200])
            return []
        data = r.json()
        return (data.get("resultList") or {}).get("result") or []
    except Exception as e:
        print("Europe PMC request failed:", e)
        return []


def best_match_europepmc(ref, items):
    if not items:
        return None, 0.0

    best = None
    best_score = 0.0

    ref_title = (ref.get("title") or "").lower()
    ref_first_author_surname = None
    if ref.get("authors"):
        ref_first_author_surname = ref["authors"][0].split()[-1].lower()

    ref_year = None
    if ref.get("year") and str(ref["year"]).isdigit():
        ref_year = int(ref["year"])

    for c in items:
        cand_title = (c.get("title") or "").lower()
        if not cand_title:
            continue

        title_score = fuzz.token_sort_ratio(ref_title, cand_title)

        cand_first_surname = None
        # firstAuthor looks like "Cladouhos TT"
        if c.get("authorString"):
            # authorString may be "Cladouhos TT, Smith J, ..."; take the first part
            first_part = c["authorString"].split(",")[0]
            cand_first_surname = first_part.split()[0].lower()

        author_score = 0
        if ref_first_author_surname and cand_first_surname:
            author_score = 100 if ref_first_author_surname == cand_first_surname else 0

        cand_year = None
        if "pubYear" in c and str(c["pubYear"]).isdigit():
            cand_year = int(c["pubYear"])

        year_bonus = 0
        if ref_year and cand_year:
            if cand_year == ref_year:
                year_bonus = 10
            elif abs(cand_year - ref_year) <= 1:
                year_bonus = 5

        score = 0.6 * title_score + 0.3 * author_score + year_bonus

        if score > best_score:
            best_score = score
            best = c

    return best, best_score


In [ ]:
import pandas as pd

results = []

for i, ref in enumerate(references, start=1):
    print(f"\n=== Reference {i} ===")
    print("Title:", ref["title"])
    print("Authors:", ref["authors"])
    print("Year:", ref["year"])

    # --- OpenAlex ---
    oa_candidates = query_openalex(ref, mailto="youremail@example.com", max_results=5)
    best_oa, score_oa = best_match_openalex(ref, oa_candidates)

    # --- Crossref ---
    cr_candidates = query_crossref(ref, mailto="youremail@example.com", max_results=5)
    best_cr, score_cr = best_match_crossref(ref, cr_candidates)

    # --- PubMed ---
    pm_candidates = query_pubmed(ref, email="youremail@example.com", max_results=5)
    best_pm, score_pm = best_match_pubmed(ref, pm_candidates)

    # --- EuropePMC ---
    ep_candidates = query_europepmc(ref, max_results=5)
    best_ep, score_ep = best_match_europepmc(ref, ep_candidates)

    # best score across the sources
    source_scores = [
        ("OpenAlex", best_oa, score_oa),
        ("Crossref", best_cr, score_cr),
        ("PubMed", best_pm, score_pm),
        ("EuropePMC", best_ep, score_ep),
    ]

    best_source = None
    best_obj = None
    best_score = 0.0

    for sname, obj, sc in source_scores:
        if obj and sc > best_score:
            best_source = sname
            best_obj = obj
            best_score = sc

    status = categorize_score(best_score)

    matched_title = None
    matched_year = None
    matched_id = None
    matched_doi = None

    if best_source == "OpenAlex" and best_obj:
        matched_title = best_obj.get("title")
        matched_year = best_obj.get("publication_year")
        matched_id = best_obj.get("id")
        ids = best_obj.get("ids") or {}
        matched_doi = ids.get("doi")
    elif best_source == "Crossref" and best_obj:
        titles = best_obj.get("title") or []
        matched_title = titles[0] if titles else None
        issued = best_obj.get("issued") or {}
        date_parts = issued.get("date-parts") or []
        if date_parts and date_parts[0]:
            matched_year = date_parts[0][0]
        matched_id = best_obj.get("URL")
        matched_doi = best_obj.get("DOI")
    elif best_source == "PubMed" and best_obj:
        matched_title = best_obj.get("title")
        matched_year = best_obj.get("pubdate", "")[:4]
        matched_id = best_obj.get("uid")
        matched_doi = best_obj.get("elocationid")  # not always a DOI
    elif best_source == "EuropePMC" and best_obj:
        matched_title = best_obj.get("title")
        matched_year = best_obj.get("pubYear")
        matched_id = best_obj.get("id")
        matched_doi = best_obj.get("doi")

    results.append(
        {
            "our_title": ref["title"],
            "our_first_author": ref["authors"][0] if ref["authors"] else None,
            "our_year": ref["year"],
            "best_source": best_source,
            "score": round(best_score, 1),
            "status": status,
            "matched_title": matched_title,
            "matched_year": matched_year,
            "matched_doi": matched_doi,
            "matched_id": matched_id,
            "score_openalex": round(score_oa, 1) if best_oa else None,
            "score_crossref": round(score_cr, 1) if best_cr else None,
            "score_pubmed": round(score_pm, 1) if best_pm else None,
            "score_europepmc": round(score_ep, 1) if best_ep else None,
        }
    )

df = pd.DataFrame(results)
df


In [ ]:
output_name = "reference_check_results.csv"
df.to_csv(output_name, index=False)
print("Saved:", output_name)

from google.colab import files
files.download(output_name)
